# 03 — Feature Engineering: the Feature Store (PRD §13, T-042)

**Purpose:** assemble the first-iteration feature store from the raw synthetic
node table through the Phase-2 pipeline — §10 windowing (60/10) → feature
Groups A–F → §12 majority labels per window — and **assert the §13 budget of
40–70 engineered features**, failing loudly outside it.

**Design notes carried from the modules:**
- Group B rolling statistics are series-internal (no train/test boundary
  crossing — §23, executable check in `tests/test_group_b.py`);
- `distance_to_subsidence_center` uses the G-5 rule: `oracle` mode here (legal
  for §10 synthetic training data where the centre is config-defined);
  inference must use `detected` mode;
- Group D features come from §8.3 on-node **summarised** vibration only.

In [1]:
import sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.features.build_feature_store import build_feature_store, feature_names
from src.features.windowing import windowing_params

REPO = Path.cwd().parent
nodes_path = REPO / "data" / "synthetic" / "synthetic_nodes.csv"
window, stride = windowing_params()
print(f"windowing: window={window} stride={stride} (§10, config-driven)")

windowing: window=60 stride=10 (§10, config-driven)


In [2]:
usecols = ["event_id", "node_id", "timestamp", "x", "y",
           "tilt_x", "tilt_y", "tilt_magnitude", "displacement", "strain",
           "vibration_rms", "vibration_peak", "battery", "RSSI", "SNR", "packet_loss",
           "anomaly_label", "risk_label", "progression_label", "fault_label"]
t0 = time.time()
raw = pd.read_csv(nodes_path, usecols=usecols)
coords = raw.groupby("node_id", sort=False)[["x", "y"]].first().reset_index()
print(f"loaded {len(raw):,} rows, {raw.event_id.nunique():,} events in {time.time()-t0:.1f}s")

loaded 1,440,000 rows, 10,000 events in 1.2s


## 1 — Assemble the feature store (Groups A–F over §10 windows)

One node per event in the gate dataset ⇒ Group C neighbourhoods are
self-only on synthetic data (spatial features are structurally exercised by
`tests/test_group_c.py` on a mesh); the honest synthetic-data semantics are
recorded rather than hidden.

In [3]:
t0 = time.time()
model, report = build_feature_store(raw, coords, center_mode="oracle")
print(f"feature store: {report.n_windows:,} windows in {time.time()-t0:.1f}s")
print(f"features: {len(report.features)}")
for group, feats in report.per_group.items():
    print(f"  {group:16s} {len(feats):2d} features")

feature store: 90,000 windows in 115.8s
features: 61
  A_physical        5 features
  B_temporal       30 features
  C_spatial         8 features
  D_vibration       7 features
  E_sensor_health   7 features
  F_physics         4 features


## 2 — The §13 budget assertion (40–70 features, inclusive)

In [4]:
n = len(report.features)
MIN_BUDGET, MAX_BUDGET = 40, 70
print(f"assembled feature count = {n}; §13 budget = [{MIN_BUDGET}, {MAX_BUDGET}]")
assert MIN_BUDGET <= n <= MAX_BUDGET, (
    f"§13 feature budget violated: {n} features, allowed {MIN_BUDGET}-{MAX_BUDGET}"
)
print("budget assertion PASSED")

assembled feature count = 61; §13 budget = [40, 70]
budget assertion PASSED


## 3 — Sanity: labels stay separate, features are finite, windows are per-series

In [5]:
from src.preprocessing.labels import validate_labels, assert_labels_separate

validate_labels(model)          # §12 vocabularies
assert_labels_separate(model)   # no collapse into one flag

feature_cols = report.features
finite_ok = model[feature_cols].apply(lambda s: np.isfinite(s.to_numpy(dtype=float)).all()).all()
print(f"all {len(feature_cols)} feature columns finite: {bool(finite_ok)}")
assert finite_ok

per_series = model.groupby(["event_id", "node_id"]).size()
print(f"windows per series: min={per_series.min()} max={per_series.max()} (expect 9 = (144-60)/10+1)")
assert (per_series == 9).all()
print(f"risk distribution in the store:\n{model['risk_label'].value_counts().to_string()}")

all 61 feature columns finite: True
windows per series: min=9 max=9 (expect 9 = (144-60)/10+1)
risk distribution in the store:
risk_label
NORMAL      61875
WARNING     18882
CRITICAL     9243


## 4 — Persist the feature-store artifacts (§33 data/features/)

In [6]:
out_dir = REPO / "data" / "features"
out_dir.mkdir(parents=True, exist_ok=True)
parquet_path = out_dir / "features_v1.parquet"
model.to_parquet(parquet_path, index=False)

report_json = {
    "feature_schema_version": "v1",
    "windowing": {"window_steps": window, "stride": stride},
    "n_windows": int(report.n_windows),
    "n_features": len(report.features),
    "features": report.features,
    "per_group": {k: len(v) for k, v in report.per_group.items()},
    "budget": {"min": MIN_BUDGET, "max": MAX_BUDGET, "ok": True},
    "source": "data/synthetic/synthetic_nodes.csv",
}
(out_dir / "feature_store_report.json").write_text(json.dumps(report_json, indent=2))
print(f"wrote {parquet_path} ({parquet_path.stat().st_size/1e6:.1f} MB)")
print(f"wrote {out_dir / 'feature_store_report.json'}")

wrote /Users/harshkumarjha/mine/data/features/features_v1.parquet (34.5 MB)
wrote /Users/harshkumarjha/mine/data/features/feature_store_report.json


## Verdict

In [7]:
print("T-042 FEATURE STORE")
print(f"  windows           : {report.n_windows:,}")
print(f"  features assembled: {len(report.features)} (budget 40-70: PASS)")
print(f"  §12 labels        : separate columns, vocabularies validated")
print(f"  artifacts         : data/features/features_v1.parquet + report")
verdict = True
assert verdict

T-042 FEATURE STORE
  windows           : 90,000
  features assembled: 61 (budget 40-70: PASS)
  §12 labels        : separate columns, vocabularies validated
  artifacts         : data/features/features_v1.parquet + report
